In [94]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [95]:
df=pd.read_csv("/content/all-data.csv")

In [96]:
data=df.copy()

In [97]:
data.head()

,Sentimant,Sentence
0,neutral,"According to Gran , the company has no plans t..."
1,neutral,Technopolis plans to develop in stages an area...
2,negative,The international electronic industry company ...
3,positive,With the new production plant the company woul...
4,positive,According to the company 's updated strategy f...


In [98]:
from sklearn.model_selection import train_test_split
x=data["Sentence"]
y=data["Sentimant"]

**Split all the labels in y among y_train and Y-test**

In [99]:
x_tmp, x_test, y_tmp, y_test = train_test_split(x, y, test_size=0.15, random_state=42, stratify=y)
x_train, x_val, y_train, y_val = train_test_split(x_tmp, y_tmp, test_size=0.1765, random_state=42, stratify=y_tmp)

**Encoding the y label**

In [100]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_val   = encoder.transform(y_val)
y_test  = encoder.transform(y_test)
print(encoder.classes_, len(x_train), len(x_val), len(x_test))

['negative' 'neutral' 'positive'] 3391 728 727


In [101]:
y_train[10::]

array([1, 1, 0, ..., 2, 2, 1])

In [102]:
x_train[::5]

,Sentence
3695,No financial or pricing details were disclosed .
2345,"According to latest information , Benefon will..."
3551,Cargotec will also move Hallberg-Ivarsson 's s...
4633,The repurchases shall decrease the distributab...
4760,"Revenue was slightly down , at x20ac 495 mill..."
...,...
4290,Viking Line has canceled some services .
2020,"In the beginning of this year , Wartsila had s..."
961,Operating profit improved by 44.0 % to ER 4.7 ...
2573,STUK today is a full service house expert in r...


In [103]:
import re

def preprocess(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-z0-9\s%.,]", " ", text)
    text = re.sub(r"(?<!\d)[.,]|[.,](?!\d)", " ", text)   # drop . and , unless inside a number
    return re.sub(r"\s+", " ", text).strip()

x_train, x_val, x_test = [s.apply(preprocess) for s in (x_train, x_val, x_test)]
print(x_train.str.split().str.len().describe())

count    3391.000000
mean       20.906222
std         9.038694
min         1.000000
25%        14.000000
50%        20.000000
75%        27.000000
max        52.000000
Name: Sentence, dtype: float64


In [104]:
x_train[::5]

,Sentence
3695,no financial or pricing details were disclosed
2345,according to latest information benefon will l...
3551,cargotec will also move hallberg ivarsson s se...
4633,the repurchases shall decrease the distributab...
4760,revenue was slightly down at x20ac 495 million...
...,...
4290,viking line has canceled some services
2020,in the beginning of this year wartsila had sec...
961,operating profit improved by 44.0 % to er 4.7 ...
2573,stuk today is a full service house expert in r...


In [105]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_len = int(np.percentile(x_train.str.split().str.len(), 99))   # likely ~60-70, not 100
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>", filters="")
tokenizer.fit_on_texts(x_train)                                   # train only -> no leakage

def prep(s): return pad_sequences(tokenizer.texts_to_sequences(s), maxlen=max_len)
x_train_pad, x_val_pad, x_test_pad = prep(x_train), prep(x_val), prep(x_test)

**All the vectors are converted into same length**

In [106]:
x_train_pad[1]

array([   0,    0,    0,    0,    0,   24, 1863,  123,  163,  643,   48,
         22,   15,  263,  206,   42,   29,   16,   31,   32,  107,    6,
       4177,   43,   61, 2838,   43,    8,    2,   58,  259,    3,   42,
         14, 4178,   43,   61, 2839,   43,    8,    2,  140,   45,    3,
         50], dtype=int32)

**Using Fast Text embedding Technique**

In [107]:
!pip install gensim

In [108]:
#import gensim.downloader as api

#fasttext_model = api.load("fasttext-wiki-news-subwords-300")

Downloaded the pre trained fasttext model whichis trained on millions of words so that no words are missed

In [109]:
#fasttext_model.save('/content/drive/MyDrive/fasttext.model')

Saved the model into the drive so that when run time is disconnected the model is saved into the drive and can be used when needed

In [110]:
from gensim.models import KeyedVectors
fasttext_model=KeyedVectors.load('/content/drive/MyDrive/fasttext.model')

In [111]:
from gensim.models import KeyedVectors
fasttext_model = KeyedVectors.load("/content/drive/MyDrive/fasttext.model")

embedding_dim = 300
word_index = tokenizer.word_index
rng = np.random.default_rng(42)
embedding_matrix = rng.normal(0, 0.05, (len(word_index) + 1, embedding_dim))
embedding_matrix[0] = 0                                           # padding index
covered = 0
for word, i in word_index.items():
    if word in fasttext_model:
        embedding_matrix[i] = fasttext_model[word]; covered += 1
print("Coverage:", covered / len(word_index))

Coverage: 0.7956212370005473


In [112]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(class_weights))

print(class_weights)

{0: np.float64(2.678515007898894), 1: np.float64(0.56095947063689), 2: np.float64(1.1848357791754018)}


checks how many of the vocabulary is understood by the FastText

It helps  understand:
- how useful your embedding matrix will be
- whether your model will perform well

**building an LSTM model without dropout**

In [113]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model_lstm = Sequential([
    Embedding(
        input_dim=len(word_index) + 1,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        input_length=100,   # same as max_len
        trainable=False
    ),
    LSTM(64, return_sequences=True, dropout=0.2, recurrent_dropout=0.2),
    LSTM(32, dropout=0.2, recurrent_dropout=0.2),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


early stopping monitors the validation loss and stops the training when the validation loss stops improving

In [114]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor='val_loss',      # watch validation loss
    patience=2,              # wait 2 epochs before stopping
    restore_best_weights=True
)

**Compling the model**

In [115]:
model_lstm.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [116]:
model_lstm.fit(
    x_train_pad,
    y_train,
    epochs=15,
    batch_size=32,
    validation_data=(x_val_pad, y_val),
    class_weight=class_weights,
    #callbacks=[early_stopping]
)

Epoch 1/15
106/106 ━━━━━━━━━━━━━━━━━━━━ 45s 319ms/step - accuracy: 0.5689 - loss: 1.0684 - val_accuracy: 0.5810 - val_loss: 0.9455
Epoch 2/15
106/106 ━━━━━━━━━━━━━━━━━━━━ 33s 316ms/step - accuracy: 0.5659 - loss: 0.9819 - val_accuracy: 0.5646 - val_loss: 0.9997
Epoch 3/15
106/106 ━━━━━━━━━━━━━━━━━━━━ 35s 330ms/step - accuracy: 0.5857 - loss: 0.9243 - val_accuracy: 0.5687 - val_loss: 0.9481
Epoch 4/15
106/106 ━━━━━━━━━━━━━━━━━━━━ 33s 312ms/step - accuracy: 0.5975 - loss: 0.9003 - val_accuracy: 0.6168 - val_loss: 0.8504
Epoch 5/15
106/106 ━━━━━━━━━━━━━━━━━━━━ 41s 311ms/step - accuracy: 0.6063 - loss: 0.8705 - val_accuracy: 0.5920 - val_loss: 0.8563
Epoch 6/15
106/106 ━━━━━━━━━━━━━━━━━━━━ 35s 327ms/step - accuracy: 0.6157 - loss: 0.8419 - val_accuracy: 0.5975 - val_loss: 0.7987
Epoch 7/15
106/106 ━━━━━━━━━━━━━━━━━━━━ 33s 311ms/step - accuracy: 0.6267 - loss: 0.7933 - val_accuracy: 0.6346 - val_loss: 0.7688
Epoch 8/15
106/106 ━━━━━━━━━━━━━━━━━━━━ 33s 314ms/step - accuracy: 0.6559 - loss: 0

In [117]:
loss, acc = model_lstm.evaluate(x_test_pad, y_test)
print("Test Accuracy:", acc)

23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 69ms/step - accuracy: 0.7455 - loss: 0.6119
Test Accuracy: 0.7455295920372009


In [118]:
import numpy as np
from sklearn.metrics import classification_report

y_pred = model_lstm.predict(x_test_pad)
y_pred = np.argmax(y_pred, axis=1)

print(classification_report(y_test, y_pred))

23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 121ms/step
              precision    recall  f1-score   support

           0       0.60      0.82      0.70        91
           1       0.84      0.79      0.82       432
           2       0.63      0.61      0.62       204

    accuracy                           0.75       727
   macro avg       0.69      0.74      0.71       727
weighted avg       0.75      0.75      0.75       727



Moving to Bi directional LSTM because LSTM is not performing well as it the data is un balanced data

In [119]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional

model_bi_lstm = Sequential([
    Embedding(
        input_dim=len(word_index) + 1,   # vocabulary size
        output_dim=embedding_dim,        # 300 for FastText
        weights=[embedding_matrix],      # pretrained FastText
        input_length=100,                # same as max_len
        trainable=False                 # keep frozen initially
    ),

    Bidirectional(LSTM(64, return_sequences=True)),
    Bidirectional(LSTM(32)),            # BiLSTM layer

    Dropout(0.5),                      # prevent overfitting

    Dense(32, activation='relu'),

    Dense(3, activation='softmax')     # 3 classes
])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [120]:
model_bi_lstm.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

Early stopping in BI lstm is not recommend because the model is under fitting

In [121]:
history = model_bi_lstm.fit(
    x_train_pad,
    y_train,
    epochs=10,
    batch_size=32,
    validation_data=(x_val_pad, y_val),
    class_weight=class_weights,   # use your tuned weights
)

Epoch 1/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 6s 21ms/step - accuracy: 0.4786 - loss: 1.0441 - val_accuracy: 0.6140 - val_loss: 0.9373
Epoch 2/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.5789 - loss: 0.9365 - val_accuracy: 0.6607 - val_loss: 0.8058
Epoch 3/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.6101 - loss: 0.8616 - val_accuracy: 0.6580 - val_loss: 0.7860
Epoch 4/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - accuracy: 0.6216 - loss: 0.8042 - val_accuracy: 0.4863 - val_loss: 0.9559
Epoch 5/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.6612 - loss: 0.7303 - val_accuracy: 0.7047 - val_loss: 0.6918
Epoch 6/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.7033 - loss: 0.6645 - val_accuracy: 0.6126 - val_loss: 0.8034
Epoch 7/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.7284 - loss: 0.6155 - val_accuracy: 0.6690 - val_loss: 0.7643
Epoch 8/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.7328 - loss: 0.5934 - val_accu

In [122]:
loss, acc = model_bi_lstm.evaluate(x_test_pad, y_test)
print("Test Accuracy:", acc)

23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7497 - loss: 0.6128
Test Accuracy: 0.7496561408042908


In [123]:
import numpy as np
from sklearn.metrics import classification_report

y_pred = model_bi_lstm.predict(x_test_pad)
y_pred = np.argmax(y_pred, axis=1)

print(classification_report(y_test, y_pred))

23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step
              precision    recall  f1-score   support

           0       0.67      0.67      0.67        91
           1       0.81      0.85      0.83       432
           2       0.63      0.57      0.60       204

    accuracy                           0.75       727
   macro avg       0.71      0.70      0.70       727
weighted avg       0.74      0.75      0.75       727



In [124]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout

model_gru = Sequential([
    Embedding(
        input_dim=len(word_index) + 1,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        input_length=100,
        trainable=False
    ),

    GRU(64, return_sequences=True, dropout=0.2, recurrent_dropout=0.2),
    GRU(32, dropout=0.2, recurrent_dropout=0.2),                 # GRU instead of LSTM

    #Dropout(0.5),

    Dense(32, activation='relu'),

    Dense(3, activation='softmax')
])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [125]:
model_gru.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [126]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

In [127]:
history = model_gru.fit(
    x_train_pad,
    y_train,
    epochs=10,
    batch_size=32,
    validation_data=(x_val_pad, y_val),
    class_weight=class_weights,
    callbacks=[early_stopping]
)

Epoch 1/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 37s 294ms/step - accuracy: 0.4170 - loss: 1.0423 - val_accuracy: 0.5838 - val_loss: 0.9261
Epoch 2/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 30s 284ms/step - accuracy: 0.5326 - loss: 0.9676 - val_accuracy: 0.6360 - val_loss: 0.8493
Epoch 3/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 31s 288ms/step - accuracy: 0.5771 - loss: 0.8958 - val_accuracy: 0.5962 - val_loss: 0.8538
Epoch 4/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 32s 303ms/step - accuracy: 0.5945 - loss: 0.8399 - val_accuracy: 0.5673 - val_loss: 0.8300
Epoch 5/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 30s 283ms/step - accuracy: 0.6087 - loss: 0.8048 - val_accuracy: 0.5797 - val_loss: 0.8120
Epoch 6/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 30s 282ms/step - accuracy: 0.6508 - loss: 0.7559 - val_accuracy: 0.6058 - val_loss: 0.8239
Epoch 7/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 30s 281ms/step - accuracy: 0.6727 - loss: 0.7109 - val_accuracy: 0.6813 - val_loss: 0.6906
Epoch 8/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 30s 280ms/step - accuracy: 0.6930 - loss: 0

In [128]:
loss, acc = model_gru.evaluate(x_test_pad, y_test)
print("Test Accuracy:", acc)

23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 53ms/step - accuracy: 0.7318 - loss: 0.6815
Test Accuracy: 0.7317743897438049


In [129]:
import numpy as np
from sklearn.metrics import classification_report

y_pred = model_gru.predict(x_test_pad)
y_pred = np.argmax(y_pred, axis=1)

print(classification_report(y_test, y_pred))

23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 78ms/step
              precision    recall  f1-score   support

           0       0.62      0.69      0.65        91
           1       0.82      0.82      0.82       432
           2       0.60      0.57      0.58       204

    accuracy                           0.73       727
   macro avg       0.68      0.69      0.68       727
weighted avg       0.73      0.73      0.73       727

